# V04 - Data Modeling: espacos, containers, views e data model

**Objetivo:** percorrer a hierarquia DMS criando um `container`, uma `view` e um `data model` temporarios, recupera-los, lista-los e excluir tudo na ordem correta.

**Hierarquia:** `space` -> `container` (armazenamento fisico) -> `view` (contrato de leitura) -> `data model` (agrupamento de views).

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v04',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Descoberta read-only do projeto
Antes de criar, observe o inventario atual do projeto.

In [ ]:
import pandas as pd

spaces = client.data_modeling.spaces.list(limit=100)
views = client.data_modeling.views.list(limit=100)
containers = client.data_modeling.containers.list(limit=100)
models = client.data_modeling.data_models.list(limit=100)

inventory = {
    'spaces': len(spaces),
    'containers': len(containers),
    'views': len(views),
    'data_models': len(models),
}
print('Inventario DMS (amostra ate 100 por tipo):', inventory)
spaces.to_pandas().head()

## 2. Grafico do inventario
Um grafico de barras ajuda a comunicar a composicao do projeto.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(list(inventory.keys()), list(inventory.values()))
ax.set_title('Inventario DMS por tipo (amostra)')
ax.set_ylabel('quantidade')
for i, v in enumerate(inventory.values()):
    ax.text(i, v, str(v), ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 3. Criar a hierarquia temporaria
Cria space, container (com `_load` para compatibilidade com o SDK 8.x), view e data model.

In [ ]:
from uuid import uuid4
from cognite.client.data_classes.data_modeling import (
    ContainerApply, ContainerId, DataModelApply, MappedPropertyApply,
    SpaceApply, Text, ViewApply, ViewId,
)

run = uuid4().hex[:8]
training_space = f'sp_ur_training_v04_{run}'
container_id = ContainerId(training_space, 'Equipment')
view_id = ViewId(training_space, 'Equipment', 'v1')

client.data_modeling.spaces.apply(SpaceApply(space=training_space, name='UR training - V04'))
client.data_modeling.containers.apply(ContainerApply._load({
    'space': training_space,
    'externalId': container_id.external_id,
    'properties': {'name': {'type': Text().dump(), 'nullable': False}},
    'usedFor': 'node',
}))
client.data_modeling.views.apply(ViewApply(
    space=training_space,
    external_id=view_id.external_id,
    version=view_id.version,
    properties={'name': MappedPropertyApply(container=container_id, container_property_identifier='name')},
))
model = client.data_modeling.data_models.apply(DataModelApply(
    space=training_space,
    external_id='EquipmentModel',
    version='v1',
    views=[view_id],
))
print('criados:', training_space, container_id.external_id, view_id.external_id, 'EquipmentModel')

## 4. Recuperar cada artefato
Confirma que cada contrato existe e e legivel.

In [ ]:
retrieved_container = client.data_modeling.containers.retrieve(container_id)
retrieved_view = client.data_modeling.views.retrieve(view_id)
retrieved_model = client.data_modeling.data_models.retrieve((training_space, 'EquipmentModel', 'v1'))
assert retrieved_container and retrieved_view and retrieved_model, 'Algum artefato nao foi recuperado.'
pd.DataFrame([
    {'tipo': 'container', 'external_id': container_id.external_id},
    {'tipo': 'view', 'external_id': view_id.external_id},
    {'tipo': 'data_model', 'external_id': 'EquipmentModel'},
])

## Mini-exercicio
- Adicione uma segunda propriedade ao container (ex.: `tag` do tipo Text) e remapeie na view.
- Crie uma `v2` do data model incluindo a mesma view e compare as versoes.

## 5. Limpeza idempotente
Exclui na ordem inversa da criacao: data model -> view -> container -> space.

In [ ]:
assert training_space.startswith('sp_ur_training_v04_')
client.data_modeling.data_models.delete((training_space, 'EquipmentModel', 'v1'))
client.data_modeling.views.delete(view_id)
client.data_modeling.containers.delete(container_id)
client.data_modeling.spaces.delete(training_space)
print('space_still_exists:', client.data_modeling.spaces.retrieve(training_space) is not None)